# Episodic Memory with Write-Time Reflection

This notebook demonstrates `ReflectiveMemory` and `JSONMemoryStore` working together
to give an `LLMAgent` episodic memory that distils a short lesson from each completed
task at write time.

Two properties are illustrated:

- **Write-time reflection.** After each task the agent records an episode.
  Before the episode is stored, `ReflectiveMemory` calls the LLM to produce a
  one-sentence lesson from the task and its outcome. That lesson is attached to the
  episode under `additional_data["reflection"]` and rendered automatically inside a
  `<reflection>` tag whenever the episode is displayed.
- **Recalled lessons.** When a new task arrives, `recall()` surfaces the three most
  recent episodes. Because each episode now carries a pre-distilled lesson, the agent
  sees not only what happened but what to do differently next time. For tasks that
  share a structural pattern — the country-name disambiguation error shown here — the
  recalled reflections provide explicit, actionable guidance that a raw episode
  transcript does not.

> **This notebook uses the [REST Countries API](https://restcountries.com/),**
> which is free and requires no authentication.

In [ ]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPI
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code in this notebook you need Ollama installed locally with its
service running. Download Ollama from https://ollama.com/download, then start the
service by running `ollama serve` in a terminal.

## Setup

In [1]:
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request


def ensure_ollama(host="http://localhost:11434", timeout=15):
    """Start Ollama if not already running and wait until responsive."""

    def _up():
        try:
            urllib.request.urlopen(f"{host}/api/tags", timeout=1)
            return True
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            return False

    if _up():
        return print(f"✓ Ollama already running at {host}")

    ollama_path = shutil.which("ollama")
    if ollama_path is None:
        for candidate in [
            "/teamspace/studios/this_studio/.local/bin/ollama",
            "/usr/local/bin/ollama",
            "/usr/bin/ollama",
        ]:
            if os.path.exists(candidate):
                ollama_path = candidate
                break
    if ollama_path is None:
        raise RuntimeError(
            "Could not find the ollama binary. Install with: "
            "curl -fsSL https://ollama.com/install.sh | sh",
        )

    print(f"Starting Ollama server ({ollama_path})...")
    subprocess.Popen(
        [ollama_path, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _up():
            return print(f"✓ Ollama up and running at {host}")
        time.sleep(0.5)

    raise RuntimeError(f"Ollama did not start within {timeout}s")


ensure_ollama()

✓ Ollama already running at http://localhost:11434


In [2]:
import json
import logging
import urllib.parse
from pathlib import Path

from tqdm.asyncio import tqdm

from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.data_structures.memory import Episode
from llm_agents_from_scratch.llms import OllamaLLM
from llm_agents_from_scratch.logger import enable_console_logging
from llm_agents_from_scratch.memory import JSONMemoryStore, ReflectiveMemory
from llm_agents_from_scratch.tools.simple_function import SimpleFunctionTool

## Defining the Tool

The [REST Countries API](https://restcountries.com/) performs a partial-name search.
When a short or ambiguous name matches more than one country, the tool returns all
matching names and asks the caller to be more specific. This behaviour mirrors the
kind of disambiguation error an agent encounters in practice and gives
`ReflectiveMemory` something concrete to learn from.

In [3]:
def get_country_facts(name: str) -> str:
    """Return key geographic and demographic facts for a country.

    If the name is ambiguous and matches multiple countries, the tool returns
    the list of matching names. Resubmit with the full official name from that
    list to get the facts you need.

    Uses the REST Countries API (https://restcountries.com/). No auth required.
    """
    encoded = urllib.parse.quote(name.strip())
    url = f"https://restcountries.com/v3.1/name/{encoded}"
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "llm-agents-from-scratch/1.0"},
    )
    try:
        with urllib.request.urlopen(req) as resp:
            matches = json.loads(resp.read())
    except urllib.error.HTTPError as e:
        if e.code == 404:  # noqa: PLR2004
            raise ValueError(
                f"No country found for '{name}'. Check the spelling.",
            ) from e
        raise
    if len(matches) > 1:
        names = [m["name"]["common"] for m in matches]
        return (
            f"'{name}' matches multiple countries: {', '.join(names)}. "
            f"Use the full official name and try again."
        )
    data = matches[0]
    return json.dumps(
        {
            "name": data["name"]["common"],
            "official": data["name"]["official"],
            "region": data["region"],
            "subregion": data.get("subregion", ""),
            "capital": data.get("capital", [""])[0],
            "population": data["population"],
            "area_km2": data.get("area", 0),
            "languages": list(data.get("languages", {}).values()),
            "currencies": [
                v["name"] for v in data.get("currencies", {}).values()
            ],
        },
    )


get_country_facts_tool = SimpleFunctionTool(func=get_country_facts)

## Creating the Agent with Memory

`JSONMemoryStore` persists episodes to a newline-delimited JSON file.
`ReflectiveMemory` wraps the store and adds an LLM call at write time: before each
episode is saved, it calls `llm.complete()` with the task and result to produce a
one-sentence lesson, then stores that lesson under `episode.additional_data["reflection"]`.
At recall time the last `n` episodes are returned, each carrying its reflection.
The agent holds the memory in its `memories` list. `TaskHandler` calls `recall` at
task start and `record` at task end automatically.

In [4]:
store = JSONMemoryStore(dir=Path("."), filename="reflective_episodes.jsonl")
llm = OllamaLLM(model="qwen3:14b", think=False)
memory = ReflectiveMemory(store=store, llm=llm, n=3)
agent = LLMAgent(llm=llm, tools=[get_country_facts_tool], memories=[memory])

## Part 1 — Building Episodic Memory Through Research

Four country lookups covering two patterns: a clean lookup (France) and three
ambiguous names (Sudan, Guinea, Niger) that each return a disambiguation error on
the first tool call, forcing the agent to refine the query and retry.

The first lookup runs with full logging so you can see the disambiguation error and
the retry in action. The remaining three run concurrently with logging silenced to
keep the output readable. After all four complete, `memory.summary()` shows the store
state with four stored episodes.

In [ ]:
enable_console_logging(logging.INFO)

task1 = Task(
    instruction=(
        "What are the key geographic and demographic facts about Sudan? "
        "Use the get_country_facts tool. Do not rely on prior knowledge."
    ),
)
result1 = await agent.run(task1)
print(result1)

In [ ]:
logging.disable(logging.INFO)

remaining_tasks = [
    Task(
        instruction=(
            "What are the key geographic and demographic "
            f"facts about {country}? "
            "Use the get_country_facts tool. "
            "Do not rely on prior knowledge."
        ),
    )
    for country in ["Guinea", "Niger", "France"]
]

results = await tqdm.gather(
    *[agent.run(t) for t in remaining_tasks],
    desc="Researching countries",
)

for country, result in zip(
    ["Guinea", "Niger", "France"],
    results,
    strict=False,
):
    print(f"**{country}**\n{result}\n")

In [ ]:
logging.disable(logging.NOTSET)

print(await memory.summary())

## The RecencyMemory Baseline

Before examining the reflections, it is worth seeing what `RecencyMemory` would have
surfaced at this point. A `RecencyMemory` stores no additional data. The episodes it
recalls disclose what happened — the agent's tool calls and final answer — but the
lesson is implicit in the narrative rather than explicitly extracted. A future agent
receiving these episodes would have to re-read the full result text and infer the
lesson for itself.

In [ ]:
print("What RecencyMemory would recall (same episodes, no reflections):\n")
recent_eps = await memory.store.read_recent(memory.n)
for ep in recent_eps:
    ep_without_reflection = Episode(
        task=ep.task,
        rollout=ep.rollout,
        result=ep.result,
        completed_at=ep.completed_at,
    )
    print(str(ep_without_reflection))
    print()

## Part 2 — Inspecting the Reflections

Now look at the same three episodes as stored by `ReflectiveMemory`. Each episode
carries a `<reflection>` tag holding the one-sentence lesson the LLM distilled at
write time. For the disambiguation tasks the lesson is direct and immediately
actionable: always use the full official country name. For the clean France lookup
the lesson captures a different kind of insight about the result, but is still
explicitly surfaced rather than buried in the result transcript.

In [ ]:
print("What ReflectiveMemory recalls (same episodes, with reflections):\n")
recent_eps = await memory.store.read_recent(memory.n)
for ep in recent_eps:
    print(str(ep))
    print()

## Part 3 — Recalled Lessons Guide a New Research Task

The query below asks about "Congo", which is ambiguous in exactly the same way as
the Sudan, Guinea, and Niger lookups in Part 1. Without any prior context, the agent
would call `get_country_facts("Congo")`, receive the disambiguation error, and retry
with the correct full name.

With the three most recent episodes recalled into its system prompt, the agent now
sees pre-distilled reflections about exactly this pattern. Check the logs below:
an agent that applies the recalled lesson calls the tool once with the specific name;
an agent that ignores the context calls it twice.

In [ ]:
task_congo = Task(
    instruction=(
        "What are the key geographic and demographic facts about Congo? "
        "Use the get_country_facts tool. Do not rely on prior knowledge."
    ),
)

print("Episodes that will be recalled for this task:\n")
recalled = await memory.store.read_recent(memory.n)
for ep in recalled:
    print(str(ep))
    print()

In [ ]:
enable_console_logging(logging.INFO)

result_congo = await agent.run(task_congo)
print(result_congo)

## Key Takeaway

`ReflectiveMemory` converts task experience into transferable lessons. Three concerns
stay cleanly separated: the `LLMAgent` runs tasks; `ReflectiveMemory` distils each
outcome into an explicit one-sentence lesson and delegates storage; `JSONMemoryStore`
persists the enriched episodes.

The comparison in Parts 1 and 2 makes the value concrete. Raw `RecencyMemory` episodes
disclose the full event log — tool calls, error messages, retries — but the lesson is
implicit in the narrative. `ReflectiveMemory` surfaces it in a single sentence
alongside the episode, so a future task can act on it without re-reading the entire
trajectory.

The separation of concerns is the same as in `RecencyMemory` and `SimilarityMemory`:
`ReflectiveMemory` decides *what* to generate, *what* to attach, and *how* to format
the recalled context for the prompt; `JSONMemoryStore` decides *how* to persist and
retrieve episodes. Swapping the store does not affect the reflection logic, and any
`BaseMemoryStore` that implements `read_recent()` works as a drop-in backend.